In [3]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
import os
from dotenv import load_dotenv

load_dotenv()

def get_chat():
    llm = HuggingFaceEndpoint(
        repo_id="meta-llama/Llama-3.1-8B-Instruct",
        huggingfacehub_api_token=os.environ["HUGGINGFACEHUB_API_TOKEN"],
        temperature=0,
    )
    return ChatHuggingFace(llm=llm)

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [5]:
class BLOG_State(TypedDict):
    topic: str
    outline: str
    blog: str

In [9]:
def get_outline(state: BLOG_State) -> BLOG_State:
    chat = get_chat()
    outline_prompt = f"Write a detailed outline for a blog post about {state['topic']}."
    outline = chat.invoke(outline_prompt)
    state['outline'] = outline.content
    return state

def get_blog(state: BLOG_State) -> BLOG_State:
    chat = get_chat()
    blog_prompt = f"Write a detailed blog post based on the following outline:\n{state['outline']}"
    blog = chat.invoke(blog_prompt)
    state['blog'] = blog.content
    return state

In [7]:
graph=StateGraph(BLOG_State)

graph.add_node('OUTLINE_NODE', get_outline)
graph.add_node('BLOG_NODE', get_blog)


graph.add_edge(START, 'OUTLINE_NODE')
graph.add_edge('OUTLINE_NODE', 'BLOG_NODE')
graph.add_edge('BLOG_NODE', END)

workflow = graph.compile()


In [12]:

input_state = {
    "topic": "The impact of artificial intelligence on modern society"
}
final_state = workflow.invoke(input_state)
print(final_state["blog"])

content="**The Rise of Artificial Intelligence: Exploring its Impact on Modern Society**\n\nAs we navigate the complexities of the 21st century, one technology has emerged as a game-changer in almost every aspect of our lives: artificial intelligence (AI). From the smartphones in our pockets to the self-driving cars on our roads, AI is revolutionizing the way we live, work, and interact with one another. But what does this mean for modern society? In this blog post, we'll delve into the benefits and challenges of artificial intelligence, its impact on the job market, and the social and ethical implications of its rise.\n\n**I. Introduction**\n\nArtificial intelligence has been a staple of science fiction for decades, but its applications in modern society have become increasingly real and profound. From virtual assistants like Siri and Alexa to AI-powered medical imaging and self-driving cars, the technology has transformed industries and improved lives. But as AI continues to advance,